In [1]:
# 损失函数，也是深度学习中主要要优化的函数，大白话就是用于衡量模型的输出和真实值之间的差距
# 基于衡量出的损失数值，在不断的训练中最小化损失就是学习的过程
import numpy as np
# 损失函数1：均方误差（MSE）
# 上章我用的W转置*x转置这种排布，然后发现对于softmax或者理解不够好
# 从本章开始采用XW这种形式，这样的话每次只用关心最里层的输出，也就是axis=-1这个维度
def mse(y_true:np.array,y_predict:np.array):
    # 1/2 * (y_true-y_pred)^2
    x = y_true-y_predict
    print(x)
    x = x**2
    print(x)
    x = np.sum(x,axis=-1)
    print(x)
    return 0.5 * np.sum((y_true-y_predict)**2,axis=-1)

y_pred = np.array([[0.7, 0.3],[0.9, 0.1]])
y_true = np.array([[0, 1],[1, 0]])

# 这个结果是2个样本各自的损失值大小

print(mse(y_true=y_true,y_predict=y_pred))

# 若要输出全部2样本的平均损失，直接去掉axis就行了
def batch_mse(y_true:np.array,y_predict:np.array):
    # 1/2 * (y_true-y_pred)^2
    x = y_true-y_predict
    print(x)
    x = x**2
    print(x)
    x = np.sum(x,axis=-1)
    print(x)
    return 0.5 * np.sum((y_true-y_predict)**2)/y_true.shape[0]

print(batch_mse(y_true=y_true,y_predict=y_pred))


[[-0.7  0.7]
 [ 0.1 -0.1]]
[[0.49 0.49]
 [0.01 0.01]]
[0.98 0.02]
[0.49 0.01]
[[-0.7  0.7]
 [ 0.1 -0.1]]
[[0.49 0.49]
 [0.01 0.01]]
[0.98 0.02]
0.24999999999999997


In [2]:
# 交叉熵损失(带batch size的版本)
def cross_entropy_loss(y_true:np.array,y_pred:np.array):
    # loss = -y_true*log(y_pred)
    # 容易得，如果是one-hot编码的标签，其实只有1的分量参与计算了，因为其他的分量都是0，这和mse不一样
    # 但是log0为无穷，所以需要数值裁剪
    delta = 1e-7
    # 若输入是一个一维向量，需要获取维度，np和pytorch都是.ndim获取，直接返回维度数 
    if y_pred.ndim == 1:
        # 相当于变成batch_size = 1
        y_true = y_true.reshape(1,y_true.size)
        y_pred = y_pred.reshape(1,y_pred.size)
    batch_size = y_true.shape[0]
    return -np.sum(y_true*np.log(y_pred+delta))/batch_size

y_pred = np.array([[0.7, 0.3],[0.9, 0.1]])
y_true = np.array([[0, 1],[1, 0]])
print(cross_entropy_loss(y_true=y_true,y_pred=y_pred))

book_t = np.array([0,0,1,0,0,0,0,0,0,0])
book_y = np.array([0.1,0.05,0.6,0.0,0.05,0.1,0,0.1,0.0,0.0])
print(cross_entropy_loss(y_true=book_t,y_pred=book_y))


0.6546664377696898
0.510825457099338


In [3]:
# 数值微分，区别于解析解比如 y=x^2导致为y'=2x在x=2时导数为4，这是精确的解析解
# 计算机上（至少在当前的学习阶段）用的是数值解，也就是近似解，这和softmax、交叉熵一样，浮点数精度会出现舍入误差
def numerical_diff(func,x):
    # 一阶微分
    # 用中心差分避免舍入误差
    clip=1e-4
    # 公式f'(x)=limit t->0 (f(x+t)-f(x-t))/2t
    return (func(x+clip)-func(x-clip))/2/clip

def f1(x):
    return 0.01*x**2 +0.1*x

# x=5导数应为0.2
print(numerical_diff(f1,5))
# x=10导数应为0.3
print(numerical_diff(f1,10))
# 能看到结果相差非常小


0.1999999999990898
0.2999999999986347


In [ ]:
# 偏导数，f(x,y) = ax+by，则对x求偏导就是a，对y求偏导就是b
# 梯度，分别对x,y各求一次偏导，然后得到的值组合起来就是梯度，他是一个向量
# 在f(x,y) = ax+by中，梯度就是(a,b)，梯度是**函数值增加**且变化方向最快的地方
# 这里是函数值增加！！！鱼书上写的是减少，这不对
# 梯度的方向是增加函数值的，所以才要梯度下降，才要减梯度；退化到导数的时候，不要被正负号迷惑
# 记住是减去就好！
# 这里涉及到一个方向导数的概念，方向导数=cosθ*梯度，当方向导数沿梯度方向的时候就是最大值
from numpy import float64


def f2(x:np.array):
    return np.sum(x**2)

def numerical_gradient(f,array:np.array):
    # 这里可以考虑一下雅可比矩阵（多维张量怎么求，可以for循环shape[0]，也可以拍平）
    x = np.array(array,dtype=np.float64, copy=True)
    shape = x.shape
    x = x.ravel()# 张量视图，比flatten返回副本要快
    gradient_array = np.zeros_like(x,dtype=np.float64) # 生成和x一样形状的0矩阵
    clip = 1e-5
    
    for i in range(x.size):
        x[i] = x[i]+clip
        g_right = f(x.reshape(shape))
        x[i] = x[i]-2*clip
        g_left = f(x.reshape(shape))
        # print(g_right,g_left)
        # 只对要求的那个变量做中心差分，其他的值计算完正好全部减掉
        gradient_array[i] = (g_right-g_left)/2/clip
        x[i] = x[i]+clip

    return gradient_array.reshape(shape)

# 一定要输入浮点数，不然会被强制截断转为整数，造成巨大的梯度
# 2-(1e-5) -> int截断直接变成2了
print(numerical_gradient(f2,np.array([[3.,4.],[3.,5.],[3,7]])))

# 可以看等高线和梯度下降的过程
import matplotlib.pyplot as plt
def gradient_descent_trace(f, x_init, lr, steps):
    """返回梯度下降每步的坐标轨迹"""
    x = np.array(x_init, dtype=np.float64)
    trace = [x.copy()]
    for _ in range(steps):
        x -= lr * numerical_gradient(f, x)
        trace.append(x.copy())
    return np.array(trace)




[[ 6.  8.]
 [ 6. 10.]
 [ 6. 14.]]


In [ ]:
# 梯度下降的函数，最简单的方式就是直接减去梯度
# Xn = X(n-1)-lr*grad
def gradient_descent(f,x,lr,steps):
    tmp_x = x
    for i in range(steps):
        grad = numerical_gradient(f,tmp_x)
        x-=lr*grad
    return tmp_x

print(gradient_descent(f2,np.array([3.,4.]),lr = 0.1,steps = 100))

# 生成等高线网格
xs = np.linspace(-5, 5, 200)
ys = np.linspace(-5, 5, 200)
X, Y = np.meshgrid(xs, ys)
Z = X**2 + Y**2
# 记录轨迹
trace = gradient_descent_trace(f2,np.array([3.,4.]),lr = 0.1,steps = 100))
plt.figure(figsize=(6, 6))
plt.contour(X, Y, Z, levels=20, cmap='Blues')          # 等高线
plt.plot(trace[:, 0], trace[:, 1], 'o-', color='tomato', markersize=4, label='gradient descent')
plt.plot(*trace[0], 'g*', markersize=12, label='start')  # 起点
plt.plot(*trace[-1], 'r*', markersize=12, label='end')   # 终点
plt.legend()
plt.title('f(x,y) = x²+y²  Gradient Descent')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.show()

[6.11110793e-10 8.14814391e-10]
